In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [62]:
ticker = "AAPL"
start_date = "2015-01-01"
end_date = "2025-01-01"

file_path = f"../data/processed/{ticker}_{start_date}_{end_date}_returns.csv"
df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
df.head()

,Open,High,Low,Close,Volume,return,log_return
Date,,,,,,,
2015-01-05,108.29,108.65,105.41,106.25,61065973.0,-0.028172,-0.028576
2015-01-06,106.54,107.43,104.63,106.26,65565949.0,0.000094,0.000094
2015-01-07,107.20,108.20,106.69,107.75,39723689.0,0.014022,0.013925
2015-01-08,109.23,112.15,108.70,111.87,58761565.0,0.038237,0.037524
2015-01-09,112.67,113.25,110.21,112.01,53452528.0,0.001251,0.001251


In [43]:
def create_split(df, train_end="2020-12-31", val_end="2022-12-31"):
    """
    Create time-aware train/validation/test splits.
    
    Args:
        df: DataFrame with datetime index
        train_end: End date for training set
        val_end: End date for validation set
    
    Returns:
        Tuple of (train_df, val_df, test_df)
    """
    # 之前做的是：先 train-test split for time series
    # 再scale（下一个函数）
    # 再create sliding-window (sequences) dataset for training and test each （不在prepare_data当中）
    # 这个函数只做第一步

    train_df = df.loc[:train_end].copy()  # loc: 按index索引（前面已经把index设为Date了）
    val_df = df.loc[train_end:val_end].copy()
    val_df = val_df.iloc[1:]  # Remove overlap (loc索引 左右闭区间)
    test_df = df.loc[val_end:].copy()
    test_df = test_df.iloc[1:]  # Remove overlap
    
    return train_df, val_df, test_df



train_end = "2020-12-31"
val_end = "2022-12-31"
train_df, val_df, test_df = create_split(df, train_end, val_end)
print("After creating train/val/test splits:")
print("Train:", train_df.shape, train_df.index.min(), train_df.index.max())
print("Val:", val_df.shape, val_df.index.min(), val_df.index.max())
print("Test:", test_df.shape, test_df.index.min(), test_df.index.max())


After creating train/val/test splits:
Train: (1510, 7) 2015-01-05 2020-12-31
Val: (503, 7) 2021-01-04 2022-12-30
Test: (501, 7) 2023-01-04 2024-12-31


In [44]:
df.columns.tolist()

['Open', 'High', 'Low', 'Close', 'Volume', 'return', 'log_return']

In [52]:
scaler = StandardScaler()

print(scaler.get_params())
# 此时它只知道自己的“出厂设置”（比如需不需要计算均值和方差），
# 但如果你尝试访问 scaler.mean_，Python 会直接抛出 AttributeError: 'StandardScaler' object has no attribute 'mean_'
# print(scaler.mean_)  

{'copy': True, 'with_mean': True, 'with_std': True}


In [53]:
# features = df.columns.tolist()
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'return', 'log_return']
target = ['Close']

# 如果你直接写 train_df_scaled = train_df，Pandas 并不会在内存里新建一份数据。
# 它只是创建了一个新的“引用”（或者叫指针），train_df_scaled 和 train_df 指向的是内存中的同一块数据。
# 如果你不加 .copy()，当你执行下一行 train_df_scaled[features] = ... 时，Pandas 会在原地（in-place）修改这块内存。
# 加上 .copy() 会在内存中开辟一块全新的独立空间。
# 这样你就能同时保留一份纯净的原始数据（用于后续的可视化对比或作为基准），和一份缩放后的数据（喂给模型）。

train_df_scaled = train_df.copy()
train_df_scaled[features] = scaler.fit_transform(train_df[features])  # 目标容器[特定列] = 转换函数( 源容器[特定列] )


In [58]:
print("=== Scaler 内部信息诊断 ===")
print(f"1. 接收特征数量 (维度): {scaler.n_features_in_}")
print(f"2. 见过的样本行数: {scaler.n_samples_seen_}")

# hasattr 是防御性编程，检查它有没有记住列名
if hasattr(scaler, 'feature_names_in_'):
    print(f"3. 记住的特征列名: {scaler.feature_names_in_}")

print("\n--- 具体参数 ---")
# 打包打印每个特征的均值和标准差
for i, name in enumerate(scaler.feature_names_in_ if hasattr(scaler, 'feature_names_in_') else range(scaler.n_features_in_)):
    print(f"特征 [{name}]: 均值(mean) = {scaler.mean_[i]:.4f}, 标准差(scale) = {scaler.scale_[i]:.4f}")

=== Scaler 内部信息诊断 ===
1. 接收特征数量 (维度): 7
2. 见过的样本行数: 1510
3. 记住的特征列名: ['Open' 'High' 'Low' 'Close' 'Volume' 'return' 'log_return']

--- 具体参数 ---
特征 [Open]: 均值(mean) = 172.0112, 标准差(scale) = 72.7174
特征 [High]: 均值(mean) = 173.7981, 标准差(scale) = 73.7892
特征 [Low]: 均值(mean) = 170.3470, 标准差(scale) = 71.8786
特征 [Close]: 均值(mean) = 172.1717, 标准差(scale) = 72.9925
特征 [Volume]: 均值(mean) = 41832935.8391, 标准差(scale) = 31096008.6075
特征 [return]: 均值(mean) = 0.0007, 标准差(scale) = 0.0267
特征 [log_return]: 均值(mean) = 0.0001, 标准差(scale) = 0.0395
